In [1]:
import numpy as np
import matplotlib.pyplot as plt
plt.ion()
plt.rcParams["image.cmap"] = "gray"
%matplotlib widget
import sys
from utils import *

# Thème Image - TP4 - Détection de contours

Ce TP est __noté__.
- Les travaux doivent être postés sur Chamilo (ou envoyés par mail à votre chargé de TP).
- Vous rendrez UN UNIQUE FICHIER au format `html` qui contiendra le compte rendu et les codes.
- Soyez attentifs à correctement commenter et indenter le code pour faciliter la lecture.
- **De manière générale, soyez attentifs à la présentation des résultats et à la 
  rédaction. Le thème de ce TP est l'image, il est donc obligatoire
  d'illustrer votre propos avec les images produites.**
- **Utiliser autant que possible les fonctions de `matplotlib` pour une présentation professionnelle de vos résultats.**

<span style="color:red">**IMPORTANT**</span>

<span style="color:red">**Une fois tous les exercices terminés, vous exporterez le notebook au format HTML. Vous nommerez le fichier</span>

MAP201-\<Code du groupe\>-TP4-\<Prenom1\>-\<Nom1\>-\<Prenom2\>-\<Nom2\>,

<span style="color:red">**où vous remplacerez \<Code du groupe\> par le code de votre groupe de TP (e.g. INF-01) et \<Prenom\> et \<Nom\> par les prénoms et noms de votre binôme.**</span>

<span style="color:red">**C'est ce fichier (ou une conversion au format PDF) que vous devrez poster sur Chamilo (MAP201 / Groupes / Travaux) au plus tard à la fin de la séance.**
</span>


# Préambule
## Utilisation des filtres

Dans la cellule ci-dessous, on a défini les fonctions`check_and_get_patch_2D(img, u, v, m)`, `get_gauss_filter_2D` et `apply_filter_2D(im, W)` codées lors du TP3. Vous pourrez les réutiliser librement dans ce TP.

Vous pouvez tester ces deux fonctions dans la cellule suivante.

Veillez à bien comprendre ces deux cellules afin de faire les exercices suivants.

In [2]:
def check_and_get_patch_2D(img, u, v, m):
    # Check
    if u - m < 0 or u + m >= img.shape[0]:
        return False, None
    
    # Check
    if v - m < 0 or v + m >= img.shape[1]:
        return False, None
    
    # Compute
    patch = np.empty((2*m+1, 2*m+1))
    for i in range(2*m+1):
        for j in range(2*m+1):
            patch[i, j] = img[u-m+i,v-m+j]
    return True, patch

# Fonction gaussienne 1D
def f(x, sig):
    return np.exp(-x**2/(2*sig**2))

# Fonction gaussienne 2D
def f_2D(x, y, sig):
    return f(x, sig)*f(y, sig)


# Calcul d'un filtre gaussien en 2D
def get_gauss_filter_2D(sigma, epsilon):
    # Trouver le plus grand entier L tel que f(L) > epsilon
    L =  int(sigma * np.sqrt(-2 * np.log(epsilon)))

    # Calculer les poids
    w = np.empty((2*L+1, 2*L+1))
    for i in range(-L,L+1):
        for j in range(-L,L+1):
            w[L+i,L+j] = f_2D(i,j,sigma)

    # Normalisation obtenir un filtre de lissage
    w /= w.sum()
    return w

def apply_filter_2D(im, W):
    fil_img = np.empty(im.shape)
    
    if np.mod(W.shape[0],2) == 0 or np.mod(W.shape[1],2) == 0:
        raise ValueError("Le filtre doit avoir une longueur impaire.")
    else:
        m = int((W.shape[0] - 1) / 2)
    
    # Calculer l'image filtree
    for u in range(im.shape[0]):
        for v in range(im.shape[1]):
            ok, patch = check_and_get_patch_2D(im, u, v, m)
            if ok:
                mean_val = np.sum(patch * W)
            else:
                mean_val = 0
            fil_img[u, v] = mean_val
    return fil_img


In [ ]:
# L'image d'origine
im1 = plt.imread('papillon.bmp')

# Filtrage avec un filtre moyen
M = np.ones([3,3]) / 9 # Filtre moyen
im2 = apply_filter_2D(im1, M)

# Filtrage gaussien
G = get_gauss_filter_2D(2.5,0.001)
im3 = apply_filter_2D(im1,G)


fig,ax = plt.subplots(1,3,figsize=(12,4))
ax[0].imshow(im1,vmin=0,vmax=255,cmap='gray')
ax[0].set_axis_off()
ax[0].set_title('Origine')
ax[1].imshow(im2,vmin=0,vmax=255,cmap='gray')
ax[1].set_axis_off()
ax[1].set_title('Fil. Moyen')
ax[2].imshow(im3,vmin=0,vmax=255,cmap='gray')
ax[2].set_axis_off()
ax[2].set_title('Fil. gaussien')



# Dérivées de l'image

**Rappels de quelques définitions** : On munit le plan du système de coordonnées habituel $(x,y)$. 
Une image continue est vue comme une fonction
$$I :
  \begin{array}{ccl}
    \mathbb{R}^2 &\to& \mathbb{R}\\
    (x,y) &\mapsto& I(x,y),
  \end{array}$$

où $I(x,y)$ correspond à l'intensité lumineuse au point $(x,y)$.
On note
$\frac{\partial I}{\partial x}$ et $\frac{\partial I}{\partial y}$
les dérivées partielles (qu'on suppose définies partout) et 
$\nabla\ I = (\frac{\partial I}{\partial x}, \frac{\partial I}{\partial y})$
le gradient de $I$. La norme du gradient
$$\|\nabla\ I\| = \sqrt{\left(\frac{\partial I}{\partial x}\right)^2 +
  \left(\frac{\partial I}{\partial y}\right)^2}$$
est donc une fonction de $\mathbb{R}^2$ dans $\mathbb{R}$.



## Exercice 1 (5 points)
Soit `im` une image numérique donnée sous la forme d'un tableau à deux dimensions
contenant des entiers de 0 à 255. L'équivalent numérique des dérivées partielles de l'image est obtenu par application
des filtres $\verb#Dx# = \begin{bmatrix}0&0&0\\-0.5&0&0.5\\0&0&0\end{bmatrix}$ pour la direction $x$ et $\verb#Dy# = \begin{bmatrix}0&0.5&0\\0&0&0\\0&-0.5&0\end{bmatrix}$ pour la direction $y$. On notera `imx` et `imy` les dérivées partielles par rapport à $x$ et $y$ respectivement. 

1. Coder deux fonctions `derive_partielle_x(im)` et `derive_partielle_y(im)` permettant de calculer `imx` et `imy`.
2. Tester le résultat sur les images `desert`, `papillon` et `sweets`.  Afficher l'image originale dans la première colonne, `imx` dans la deuxième et `imy` dans la troisième.
3. Charger l'image `modele`, calculer `imy` et afficher les deux. Expliquer pourquoi cette dernière est nulle.


In [4]:
# Votre reponse à la question 1.

def derivee_partielle_x(im):
    ### A COMPLETER ###
    return imx

def derivee_partielle_y(im):
    ### A COMPLETER ###
    return imy


In [ ]:
# Votre reponse à la question 2.

desert = np.array(plt.imread('desert.bmp'))
papillon = np.array(plt.imread('papillon.bmp'))
sweets = np.array(plt.imread('sweets.bmp'))

images = (desert, papillon, sweets)
fig,ax = plt.subplots(len(images),3,figsize=(len(images)*3,9))

for i, im in enumerate(images):
	### A COMPLETER ###

In [ ]:
# Votre reponse à la question 3.

modele = np.array(plt.imread('modele.bmp'))

### A COMPLETER ###


## Exercice 2 (5 points)

Pour chaque pixel `(u,v)`, le norme du gradient est définie par 
$$\verb#imn#(u,v) = \sqrt{\verb#imx#(u,v)^2 + \verb#imy#(u,v)^2}.$$


1. Coder la fonction `norme_gradient(im)` pour qu'elle renvoie la norme `imn` du gradient de l'image `im`, selon la formule précédente.
2. Tester le résultat et visualiser la norme du gradient (sur l'image de votre choix).
3. Si nécessaire, améliorer la visualisation en corrigeant le contraste de l'image `imn`. Vous expliquerez la méthode choisie.


In [ ]:
# Votre réponse à la question 1. 

def norme_gradient(im):
    ### A COMPLETER ###
    return imn

# Votre réponse à la question 2. 

### A COMPLETER ###

# Votre réponse à la question 3.

### A COMPLETER ###



# Détection de contours par seuillage simple

Une première méthode de détection des contours consiste à les définir
comme les points de l'image où la norme du gradient dépasse un certain `seuil`.
L'image `imc` contenant les contours est définie par
$$
  \tt
  \forall (u,v), imc(u,v) = \left\lbrace
    \begin{array}{ccc}
      \tt 0 &\text{\tt si}&\tt  imn(u,v) < seuil,\\
      \tt 255 &\text{\tt sinon,}
    \end{array}
  \right. 
$$
où `imn` est l'image contenant la norme du gradient de l'image `im`.

## Exercice 3 (5 points)
1. Compléter la fonction `contours_seuil(im, seuil)` de manière à ce qu'elle renvoie l'image `imc` telle que définie ci-dessus. 
2. Puis tester la fonction, par exemple sur `sweets` et avec plusieurs valeurs de seuil (i.e. faire varier la valeur du paramètre `seuil` pour évaluer son influence sur le résultat). Afficher les image de contours pour différentes valeurs du seuil et  commenter.

In [ ]:
# Votre réponse à la question 1.
# Compléter l'une ou l'autre des deux fonctions suivantes. La première effectue une double boucle sur les pixels. La seconde n'a pas recours aux boucles.

def contours_seuil_boucle(im, seuil):
		imn = norme_gradient(im)
		imc = np.zeros_like(im)
		for i in range(imc.shape[0]):
			for j in range(imc.shape[1]):
					### A COMPLETER ###
                                        
		return imc

def contours_seuil(im, seuil):
    imn = norme_gradient(im)
    imc = np.zeros_like(im)
    
    ### A COMPLETER ###

    return imc

# Votre réponse à la question 2.
# Utiliser l'une ou l'autre des fonctions définies ci-dessus.

im = np.array(plt.imread('sweets.bmp'))
seuils = (10,40,100)
fig, ax = plt.subplots(1,1+len(seuils),figsize=(3*(1+len(seuils)),3))
ax[0].imshow(im)
ax[0].set_axis_off()
ax[0].set_title("Original")
for i,seuil in enumerate(seuils):
    imc = contours_seuil_boucle(im, seuil)
    ax[i+1].imshow(imc)
    ax[i+1].set_axis_off()
    ax[i+1].set_title(f"Seuil: {seuil}")


## Exercice 4 (5 points)
Si l'image est bruitée, la détection de contours par seuillage de la norme du gradient fonctionne moins bien.
1. Illustrer cela avec les images `penguin-clean` et `penguin-noise`.

Une solution consiste à appliquer un filtre de lissage (par exemple gaussien) avant de calculer la norme du gradient, de manière à ne garder que les variations les plus significatives.

2. Appliquer un filtre gaussien (avec $\sigma=2$) puis calculer la norme du gradient et essayer de détecter les contours.
Vous pouvez faire ces opérations et afficher le résultat en une ligne :

`contours_seuil(apply_filter_2D(im, get_gauss_filter_2D(sigma,0.001)),seuil)`

Dans la commande précédente:
- `get_gauss_filter_2D(sigma,0.001)`permet de calculer un filtre gaussien de paramètre `sigma`.
- `apply_filter_2D` applique le filtre gaussien à l'image.
- `contours_seuil` applique la détection de contours par seuillage avec le seuil `seuil`.

Faire varier la valeur de `seuil` pour essayer d'améliorer la détection des contours.

In [ ]:
from scipy.signal import convolve2d

img = np.array(plt.imread("penguin-clean.bmp"))
img_noise = np.array(plt.imread("penguin-noise.bmp"))

seuils = (10,40,100) 

# Votre réponse à la question 1.
# Faire une grille subplots de taille 3x4 (3 lignes et 4 colonnes). Vous complèterez la dernière ligne à la question2.

fig, ax = ### A COMPLETER ###

for i,seuil in enumerate(seuils):
		### A COMPLETER ###

		# Votre réponse à la question 2.
		# Remarque : Si les calculs sont trop longs, il est possible de les accélérer (un peu) en remplaçant 
		# apply_filter_2D(im,W) par convolve2d(im,W,mode='same')

		### A COMPLETER ###
  


# Exercice 5 (Bonus)
Comparez l'efficacité du filtre gaussien avec celle d'un filtre moyen de même taille.
Vous pourrez tester différentes tailles de filtre, différentes valeurs de $\sigma$, de `seuil`.
